# Multimodal Deepfake Detection Framework
## Phase I: Video Deepfake Detection with Reproducible Evaluation

This notebook implements the complete pipeline:
1. Face extraction from DFDC video dataset
2. XceptionNet training for binary deepfake classification
3. Evaluation with standardized metrics (AUC, EER, F1)
4. GradCAM explainability heatmaps
5. Model card generation

**Hardware**: NVIDIA GPU (tested on A10G / ml.g5.xlarge)

**Dataset**: Deepfake Detection Challenge (DFDC) — parts 00–03

---
## 1. Setup & Environment Verification

In [ ]:
import sys
import os

# Set project root — adjust if needed
PROJECT_ROOT = "/home/ec2-user/SageMaker/multimodal-deepfake-detection"
os.environ["PROJECT_ROOT"] = PROJECT_ROOT
sys.path.insert(0, PROJECT_ROOT)

import torch
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be very slow.")

In [ ]:
# Verify dataset is downloaded
from pathlib import Path
from src.config import RAW_DATA_DIR

parts = sorted(RAW_DATA_DIR.glob("part_*"))
print(f"Found {len(parts)} data parts:")
for p in parts:
    videos = list(p.glob("*.mp4"))
    print(f"  {p.name}: {len(videos)} videos")

if len(parts) == 0:
    print("\nERROR: No data found. Follow STEP_BY_STEP_INSTRUCTIONS.md Part 4 to download the dataset.")

---
## 2. Face Extraction

Extract and align faces from video frames using MTCNN.

**Expected time: 2–4 hours** for 4 parts (~1200 videos). You can start with fewer parts for a quick test.

In [ ]:
from src.face_extraction import extract_all_faces
from src.config import FACES_DIR

# Extract faces from parts 0-3
# To test quickly, change to parts=[0] (just 1 part, ~30-45 min)
total_faces = extract_all_faces(parts=[0, 1, 2, 3])

In [ ]:
# Verify extraction results
real_count = sum(1 for _ in (FACES_DIR / "real").rglob("*.jpg")) if (FACES_DIR / "real").exists() else 0
fake_count = sum(1 for _ in (FACES_DIR / "fake").rglob("*.jpg")) if (FACES_DIR / "fake").exists() else 0

print(f"\nExtracted faces:")
print(f"  Real: {real_count:,}")
print(f"  Fake: {fake_count:,}")
print(f"  Total: {real_count + fake_count:,}")
print(f"  Ratio (fake/real): {fake_count/max(real_count,1):.2f}")

---
## 3. Model Training

Fine-tune XceptionNet (pretrained on ImageNet) for binary deepfake classification.

**Expected time: 2–3 hours** on A10G for 20 epochs with early stopping.

In [ ]:
from src.train import train

model, history, split_info, test_loader = train()

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs, history['train_loss'], label='Train')
axes[0].plot(epochs, history['val_loss'], label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history['train_acc'], label='Train')
axes[1].plot(epochs, history['val_acc'], label='Val')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs, history['val_auc'], label='Val AUC', color='green')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('AUC')
axes[2].set_title('Validation AUC')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(Path(PROJECT_ROOT) / 'results' / 'metrics' / 'training_curves.png'), dpi=150)
plt.show()

---
## 4. Evaluation

Evaluate on held-out test set with standardized metrics: AUC, EER, F1, precision, recall.

In [ ]:
from src.evaluate import evaluate

results = evaluate(test_loader)

---
## 5. GradCAM Explainability

Generate heatmaps showing which facial regions the model attends to when making predictions.
These serve as explainability artifacts for human reviewers.

In [ ]:
from src.evaluate import load_best_model
from src.gradcam import generate_heatmaps

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, _ = load_best_model(device)

generate_heatmaps(model, test_loader, num_samples=50, device=device)

---
## 6. Review Results

Display sample heatmaps and final metrics.

In [ ]:
from src.config import HEATMAPS_DIR, METRICS_DIR
from IPython.display import display, Image as IPImage

# Show ROC curve
print("ROC Curve:")
display(IPImage(filename=str(METRICS_DIR / 'roc_curve.png')))

print("\nConfusion Matrix:")
display(IPImage(filename=str(METRICS_DIR / 'confusion_matrix.png')))

In [ ]:
# Show first 6 heatmap samples
heatmap_files = sorted(HEATMAPS_DIR.glob("*.png"))[:6]
print(f"\nShowing {len(heatmap_files)} sample GradCAM heatmaps:\n")
for hf in heatmap_files:
    display(IPImage(filename=str(hf)))

In [ ]:
# Print final summary
import json

with open(METRICS_DIR / 'evaluation_report.json') as f:
    report = json.load(f)

print("=" * 50)
print("FINAL RESULTS SUMMARY")
print("=" * 50)
print(f"Model:       {report['model_name']}")
print(f"Test AUC:    {report['test_auc']:.4f}")
print(f"Test EER:    {report['test_eer']:.4f}")
print(f"Test F1:     {report['test_f1']:.4f}")
print(f"Precision:   {report['test_precision']:.4f}")
print(f"Recall:      {report['test_recall']:.4f}")
print(f"Accuracy:    {report['test_accuracy']:.4f}")
print(f"Test Samples: {report['num_test_samples']}")
print("=" * 50)
print("\nPipeline complete! Results saved to results/ directory.")
print("Next: push to GitHub (see STEP_BY_STEP_INSTRUCTIONS.md Part 6)")